# Dilution of Meaning: Multi-Text Experiment

This notebook scales the iterative rewriting experiment to process all text files in the `texts/` directory. It also introduces semantic tracking using embeddings and PCA (Principal Component Analysis) to visualize how the meaning of each text "drifts" across iterations.

## Environment Setup

First, we install and import the necessary libraries. This includes `transformers` for the LLM, `sentence-transformers` for embeddings, and `scikit-learn` for PCA.

In [1]:
%pip install -q sentence-transformers scikit-learn matplotlib tqdm mlx-lm


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import glob
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
import time
import nltk
from nltk.tokenize import sent_tokenize
from mlx_lm import load, generate, batch_generate
from matplotlib.collections import LineCollection

# nltk.download('punkt_tab')

# Set device
device = "cpu"
if torch.backends.mps.is_available():
    device = torch.device("mps")

if torch.cuda.is_available():
    device = toch.device("cuda")

print(device)

mps


## Model Initialization

We initialize two models:
1.  **Generation Model**: The larger LLM used for rewriting (`google/gemma-3n-E2B-it`).
2.  **Embedding Model**: A specialized model for generating semantic vectors (`all-MiniLM-L6-v2`).

In [88]:
# Large Language Model for Generation
#gen_model_id = "Qwen/Qwen3-0.7B"
# gen_model_id = "Qwen/Qwen3.5-0.8B" 
# sum_model_id = "Qwen/Qwen3.5-0.8B"
gen_model_id = "mlx-community/Qwen3-0.6B-4bit-DWQ-053125" 
sum_model_id = "mlx-community/Qwen3-0.6B-4bit-DWQ-053125" 

print(f"Loading Generation Model: {gen_model_id}...")
# gen = pipeline("text-generation", model=gen_model_id, device=device)
gen, gen_tokenizer = load(gen_model_id)
#print(f"Loading Summarization Model: {gen_model_id}...")
#summarize = pipeline("text-generation", model=sum_model_id, device=device)
summarize, sum_tokenizer = load(sum_model_id)
# Sentence Transformer for Embeddings
print("Loading Embedding Model...")
embed_model = SentenceTransformer('all-mpnet-base-v2', device=device)


Loading Generation Model: mlx-community/Qwen3-0.6B-4bit-DWQ-053125...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [106]:
import os
from tqdm import tqdm

iterations = 200
batch_size = 8

# Define the directory where you want to save the files
output_dir = "./iteration_outputs"
trimmed_dir = "./trimmed_outputs"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(trimmed_dir, exist_ok=True)

# Optional: Keep a main log file for the experiment overview
with open(os.path.join(output_dir, "_experiment_log.txt"), "w", encoding='utf-8') as f:
    f.write("Dilution of Meaning//Semantic drift experiment type 2\n")

texts = [
    "For sale: Baby boots, never worn.",
    "The nine-year-old throws the iPad across the room, “I’m too old for this shit!”",
    "Under the tattered canopy, they clung to the threads stitched into the earth, praying that the burning cyclone wouldn’t suck them away from the abyss they now called home.",
    "He hugged his wife and boy tight, then looked up at the photo on the wall and stood at attention in his dress uniform to salute the father he barely knew.",
    "The moment I really knew it was goodbye was when I saw her kiss my best friend.",
    "When the door opens, it’s the face on the old milk carton in front of her, slightly wrinkly now but so is she.",
    "Now hiring: 7-day light-speed space exploration tour guide; job type: full-time; requirement: must be 18 years old; project duration: 10,500 years.",
    "One stoplight away from crashing her pickup into a tree to shield a young bystander from a swerving truck—and, indirectly, saving eighteen lives from a hobbling gunman’s rage years later—she was ultimately deprived of the chance by a TikToking pedestrian dancing in the street."
]

results = []

for i in tqdm(range(1, iterations + 1), desc="Processing Iterations"):
    
    # --- GENERATION STEP ---
    generate_prompts = [
        gen_tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': 'Write a scene of story of around 500 words based on the following prompt: '},
                {"role": "user", "content": current_text}
            ],
            add_generation_prompt=True,
            enable_thinking=False
        )
        for current_text in texts
    ]
    
    # Generate for the whole batch
    regenerated_outputs = batch_generate(gen, gen_tokenizer, prompts=generate_prompts, max_tokens=10000).texts
    results.append(regenerated_outputs)
    
    # --- SUMMARIZATION STEP ---
    summarize_prompts = [ 
        sum_tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': 'Write a one sentence summary that would go with this scene, capturing the main plot, theme, genre, and other relevant information.'},
                {'role': 'user', 'content': gen_text}
            ],
            add_generation_prompt=True,
            enable_thinking=False
        )
        for gen_text in regenerated_outputs
    ]
    
    # Summarize the whole batch
    summarized = batch_generate(summarize, sum_tokenizer, prompts=summarize_prompts, max_tokens=512).texts
    
    # --- EMBEDDING STEP ---
    # Pass the list of outputs directly to the encoder
    new_embeddings = embed_model.encode(regenerated_outputs, batch_size=batch_size)
    
    # --- RECORD KEEPING ---
    # Iterate through the batch to create unique files for each item
    for j, (orig_text, gen_text, summary) in enumerate(zip(texts, regenerated_outputs, summarized)):
        
        # Define the specific file paths for this iteration AND this item
        iteration_file = os.path.join(output_dir, f"iteration_{i:03d}_item_{j:03d}.txt")
        trimmed_file = os.path.join(trimmed_dir, f"iteration_{i:03d}_item_{j:03d}_trimmed.txt")
        
        # 1. Write the detailed file with prompts, outputs, and summaries
        with open(iteration_file, "w", encoding='utf-8') as f:
            f.write(f"=== ITERATION {i} | ITEM {j} ===\n\n")
            f.write(f"--- Input Prompt ---\n{orig_text}\n\n")
            f.write(f"--- Generated Text ---\n{gen_text}\n\n")
            f.write(f"--- Summary (Next Prompt) ---\n{summary}\n")
            
        # 2. Write the trimmed file containing ONLY the generated text
        with open(trimmed_file, "w", encoding='utf-8') as f:
            f.write(gen_text)
    
    # Update the texts list for the next iteration loop using the new summaries
    texts = summarized

Processing Iterations: 100%|██████████| 200/200 [11:37<00:00,  3.49s/it]


# Cosine Distance Graphs

In [107]:
import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# Ensure NLTK data is available
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize

def get_file_vector(file_path, model):
    """
    Reads a text file, splits it into sentences, embeds them,
    and returns the mean vector of all sentence embeddings.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        
        if not text.strip():
            return None

        # Split text into sentences
        sentences = sent_tokenize(text)
        
        if not sentences:
            return None

        # Generate embeddings for all sentences
        sentence_embeddings = model.encode(sentences, convert_to_numpy=True, show_progress_bar=False)
        
        # Aggregate: Mean pooling to create one vector per file
        file_vector = np.mean(sentence_embeddings, axis=0)
        return file_vector

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def calculate_file_distances(directory_path, model_name='all-MiniLM-L6-v2'):
    print(f"Loading model: {model_name}...")
    model = SentenceTransformer(model_name)
    print("Model loaded.\n")

    file_paths = sorted(glob.glob(os.path.join(directory_path, '*_trimmed.txt')))
    
    if not file_paths:
        print("No trimmed .txt files found in the directory.")
        return

    # Group files by item number using regex
    item_groups = defaultdict(list)
    for path in file_paths:
        match = re.search(r'item_(\d+)', os.path.basename(path))
        if match:
            item_id = match.group(1)
            item_groups[item_id].append(path)

    print(f"Found {len(item_groups)} distinct items to process.")

    # Process each item group independently
    for item_id, paths in item_groups.items():
        print(f"\n{'='*40}")
        print(f"Processing Item {item_id} ({len(paths)} iterations found)")
        print(f"{'='*40}")
        
        file_vectors = []
        valid_file_names = []
        iteration_labels = []

        for path in sorted(paths): 
            vector = get_file_vector(path, model)
            if vector is not None:
                file_vectors.append(vector)
                
                basename = os.path.basename(path)
                valid_file_names.append(basename)
                
                iter_match = re.search(r'iteration_(\d+)', basename)
                iter_label = iter_match.group(1) if iter_match else basename
                iteration_labels.append(f"Iter {iter_label}")

        if len(file_vectors) < 2:
            print(f"Item {item_id} needs at least 2 valid files. Skipping.")
            continue

        all_vectors = np.array(file_vectors)
        distance_matrix = cdist(all_vectors, all_vectors, metric='cosine')

        # Visualize the distance matrix
        visualize_distance_matrix(distance_matrix, iteration_labels, item_id)

        # Visualize smoothed semantic drift
        visualize_semantic_drift(distance_matrix, iteration_labels, item_id, window=10)

def visualize_distance_matrix(distance_matrix, labels, item_id):
    """
    Visualize the cosine distance matrix as a heatmap for a specific item.
    """
    fig_size = min(max(8, len(labels) * 0.4), 16)
    plt.figure(figsize=(fig_size, fig_size * 0.8))
    
    im = plt.imshow(distance_matrix, cmap='YlGnBu', aspect='auto')
    
    step = 1 if len(labels) <= 20 else max(1, len(labels) // 20)
    ticks = np.arange(0, len(labels), step)
    tick_labels = [labels[i] for i in ticks]
    
    plt.xticks(ticks, tick_labels, rotation=45, ha='right', fontsize=8)
    plt.yticks(ticks, tick_labels, fontsize=8)
    
    cbar = plt.colorbar(im)
    cbar.set_label('Cosine Distance', rotation=270, labelpad=20)
    
    plt.title(f'Cosine Distance Heatmap - Item {item_id}', fontsize=14, pad=20)
    
    plt.tight_layout()
    filename = f'cosine_distance_heatmap_item_{item_id}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Heatmap saved as '{filename}'")
    plt.close()

def moving_average(data, window_size):
    """Calculates the rolling average over a specific window size."""
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

def visualize_semantic_drift(distance_matrix, labels, item_id, window=10):
    """
    Visualize the drift using anchored distance and step-wise velocity,
    applying a rolling average to smooth the graphs.
    """
    n = len(labels)
    if n < 2:
        return

    # Raw Data
    cumulative_drift = distance_matrix[0, :]
    step_velocity = [distance_matrix[i, i-1] for i in range(1, n)]
    velocity_labels = [f"{labels[i-1]} → {labels[i]}" for i in range(1, n)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # --- Plot 1: Cumulative Drift ---
    # Plot raw data faintly
    ax1.plot(range(n), cumulative_drift, marker='o', linewidth=1, markersize=3, color='blue', alpha=0.2, label='Raw Data')
    
    # Plot smoothed data if enough points exist
    if n >= window:
        smoothed_cum = moving_average(cumulative_drift, window)
        # Shift the x-axis to align the smoothed line correctly with the trailing window
        x_smooth_cum = range(window - 1, n)
        ax1.plot(x_smooth_cum, smoothed_cum, linewidth=3, color='darkblue', label=f'{window}-Iter Moving Average')

    ax1.set_xlabel('Iteration Timeline', fontsize=12)
    ax1.set_ylabel('Distance from Iteration 1', fontsize=12)
    ax1.set_title(f'Cumulative Semantic Drift - Item {item_id}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    step1 = max(1, n // 20)
    ax1.set_xticks(np.arange(0, n, step1))
    ax1.set_xticklabels([labels[i] for i in range(0, n, step1)], rotation=45, ha='right', fontsize=8)
    ax1.set_ylim(bottom=0)

    # --- Plot 2: Velocity of Semantic Drift ---
    # Plot raw data faintly
    ax2.plot(range(len(step_velocity)), step_velocity, marker='s', linewidth=1, markersize=3, color='green', alpha=0.2, label='Raw Data')
    
    # Plot smoothed data if enough points exist
    if len(step_velocity) >= window:
        smoothed_vel = moving_average(step_velocity, window)
        x_smooth_vel = range(window - 1, len(step_velocity))
        ax2.plot(x_smooth_vel, smoothed_vel, linewidth=3, color='darkgreen', label=f'{window}-Iter Moving Average')

    ax2.set_xlabel('Transition Step', fontsize=12)
    ax2.set_ylabel('Distance to Previous Iteration', fontsize=12)
    ax2.set_title(f'Step-wise Velocity (Change Rate) - Item {item_id}', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    step2 = max(1, len(step_velocity) // 20)
    ax2.set_xticks(np.arange(0, len(step_velocity), step2))
    ax2.set_xticklabels([velocity_labels[i] for i in range(0, len(step_velocity), step2)], rotation=45, ha='right', fontsize=8)
    ax2.set_ylim(bottom=0)

    plt.tight_layout()
    filename = f'semantic_drift_velocity_smoothed_item_{item_id}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Smoothed velocity graph saved as '{filename}'")
    plt.close()

if __name__ == "__main__":
    TARGET_DIRECTORY = "./trimmed_outputs"
    calculate_file_distances(TARGET_DIRECTORY)

Loading model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.

Found 8 distinct items to process.

Processing Item 000 (200 iterations found)
Heatmap saved as 'cosine_distance_heatmap_item_000.png'
Smoothed velocity graph saved as 'semantic_drift_velocity_smoothed_item_000.png'

Processing Item 001 (200 iterations found)
Heatmap saved as 'cosine_distance_heatmap_item_001.png'
Smoothed velocity graph saved as 'semantic_drift_velocity_smoothed_item_001.png'

Processing Item 002 (200 iterations found)
Heatmap saved as 'cosine_distance_heatmap_item_002.png'
Smoothed velocity graph saved as 'semantic_drift_velocity_smoothed_item_002.png'

Processing Item 003 (200 iterations found)
Heatmap saved as 'cosine_distance_heatmap_item_003.png'
Smoothed velocity graph saved as 'semantic_drift_velocity_smoothed_item_003.png'

Processing Item 004 (200 iterations found)
Heatmap saved as 'cosine_distance_heatmap_item_004.png'
Smoothed velocity graph saved as 'semantic_drift_velocity_smoothed_item_004.png'

Processing Item 005 (200 iterations found)
H